In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.7 Time Evolution and the Schrödinger Equation

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.7",
    title="Time Evolution and the Schrödinger Equation",
    blurb="Setting the state in motion. The last postulate said the state evolves "
    "by the Schrödinger equation; here we solve it. The Hamiltonian generates a "
    "unitary flow, U(t)=e^{−iHt/ℏ}, under which energy eigenstates merely turn their "
    "phase and everything else beats between them. In the two-state system the motion "
    "is visible: a spin precesses, a driven qubit oscillates between its levels — the "
    "dynamics behind magnetic resonance, atomic clocks, and quantum gates.",
    difficulty="advanced",
    estimate="160–195 min",
)

## Notebook overview

The fifth postulate ([§6.5](postulates.ipynb)) was stated and then set aside: between measurements, the state evolves by
the Schrödinger equation $i\hbar\,\partial_t|\psi\rangle=H|\psi\rangle$. This notebook delivers on that
deferral. It solves the equation, and in doing so reveals the structure of *all* quantum dynamics —
which turns out to be simpler, and more beautiful, than the equation first suggests.

The solution is a single operator. For a time-independent Hamiltonian the state at time $t$ is
$|\psi(t)\rangle=U(t)|\psi(0)\rangle$ with the **time-evolution operator** $U(t)=e^{-iHt/\hbar}$ —
the exponential map of [§6.2](operators-spectral-theorem.ipynb), the prime function-of-operator of [§6.3](dirac-notation-spectral-decomposition.ipynb). Because $H$ is Hermitian, $U(t)$
is **unitary**, so the norm and hence total probability are conserved, exactly as a state evolution
must guarantee. And in the Hamiltonian's own eigenbasis the flow is *trivial*: each energy eigenstate
merely rotates its phase, $|n\rangle\to e^{-iE_nt/\hbar}|n\rangle$. An energy eigenstate is therefore
**stationary** — every probability frozen — and a general state is a sum of such phases beating
against one another at the Bohr frequencies $(E_n-E_m)/\hbar$. *All* dynamics is this interference of
phases, which is why the central task of quantum mechanics is to find the energy eigenstates: once $H$
is diagonalized, time evolution is free.

In the two-state system the abstract flow becomes visible motion. A spin in a magnetic field
**precesses** — its expectation vector rotates about the field at the Larmor frequency, a classical
motion emerging from unitary evolution — and a driven two-level system undergoes **Rabi oscillations**,
cycling between its levels. These are not toys: they are the physics of nuclear magnetic resonance and
MRI, of atomic clocks, and of the gates that drive a qubit in a quantum computer. We close with
**Ehrenfest's theorem**, $d\langle A\rangle/dt=(i/\hbar)\langle[H,A]\rangle$, which says exactly which
quantities are conserved — those commuting with $H$ ([§6.6](pauli-uncertainty.ipynb)) — and bridges back to the Poisson brackets
of Volume II.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — `scipy.linalg.expm` for $U(t)$ alongside the transparent
spectral construction $U=\sum_n e^{-iE_nt/\hbar}|n\rangle\langle n|$, `numpy.linalg.eigh` for the
energy eigenbasis, and `numpy.vdot` for the expectation traces $\langle A\rangle(t)$.

> **How to read the checks.** Each exercise closes with a `validate` call: $U(t)$ solving $i\hbar
> \dot U=HU$ and unitary; an energy eigenstate acquiring only a phase; a superposition beating at the
> Bohr frequency; the Larmor traces $\langle S_x\rangle=\tfrac12\cos\omega t$, $\langle S_y\rangle=
> -\tfrac12\sin\omega t$; resonant Rabi $P_e=\sin^2(\Omega t/2)$ and its off-resonant shrinking; and
> Ehrenfest's theorem with $\langle S_z\rangle$ conserved. A ✓ is strong evidence; a ✗ is a prompt to
> *locate the discrepancy*.
>
> **Conventions and scope.** We set $\hbar=1$ (restored in formulas where it clarifies). The Pauli
> matrices are those of [§6.6](pauli-uncertainty.ipynb), with spin operators $S=\tfrac12\boldsymbol\sigma$. The Larmor Hamiltonian
> is $H=-\gamma\,\mathbf B\cdot\mathbf S$; the driven two-level Hamiltonian is $H=\tfrac{\delta}{2}
> \sigma_z+\tfrac{\Omega}{2}\sigma_x$ with detuning $\delta$ and coupling $\Omega$. The Bloch-sphere
> geometry of precession is [§6.8](bloch-sphere-entanglement.ipynb); continuous-system dynamics (wave packets) is [§6.10](schrodinger-on-a-computer.ipynb)–[§6.13](scattering-tunneling.ipynb); driven
> transitions in many levels is [§6.24](time-dependent-perturbation.ipynb). See Sakurai & Napolitano (§2.1, §2.3); the Feynman Lectures Vol.
> III; and Notebooks [§6.2](operators-spectral-theorem.ipynb) (the exponential map), [§6.3](dirac-notation-spectral-decomposition.ipynb) (functions of operators), [§6.5](postulates.ipynb) (the postulate), [§6.6](pauli-uncertainty.ipynb)
> (commutators, conservation), and Volume II (Poisson brackets).

## Theory in brief

### The Schrödinger equation and the time-evolution operator

The dynamics postulate is $i\hbar\,\partial_t|\psi\rangle=H|\psi\rangle$, with $H$ the Hamiltonian
(the energy observable, Hermitian). For time-independent $H$ the solution is

```{math}
:label: eq-schrodinger
|\psi(t)\rangle=U(t)|\psi(0)\rangle,\qquad U(t)=e^{-iHt/\hbar},\qquad i\hbar\,\dot U=HU,\quad U^{\dagger}U=I .
```

$U(t)$ is the exponential map of [§6.2](operators-spectral-theorem.ipynb) / the function-of-operator of [§6.3](dirac-notation-spectral-decomposition.ipynb), and it is **unitary** (because
$H$ is Hermitian), so the norm and total probability are conserved.

### The energy eigenbasis and stationary states

In the eigenbasis of $H$, with $H|n\rangle=E_n|n\rangle$, the operator is $U(t)=\sum_n e^{-iE_nt/\hbar}
|n\rangle\langle n|$ ([§6.3](dirac-notation-spectral-decomposition.ipynb)). An **energy eigenstate** evolves by a mere phase,

```{math}
:label: eq-stationary
|\psi(t)\rangle=e^{-iE_nt/\hbar}|n\rangle\ \text{(stationary)},\qquad |\psi\rangle=\sum_n c_n|n\rangle\ \to\ \sum_n c_n e^{-iE_nt/\hbar}|n\rangle ,
```

so all its probabilities are time-independent. A general state's amplitudes **beat** at the Bohr
frequencies $(E_n-E_m)/\hbar$ — all dynamics is this interference of phases. This is why diagonalizing
$H$ is the central task: once done, time evolution is free.

### Spin precession (Larmor)

A spin in a uniform field has $H=-\gamma\,\mathbf B\cdot\mathbf S$; for $\mathbf B$ along $z$,
$H=-\tfrac{\omega}{2}\sigma_z$ with $\omega=\gamma B$ the Larmor frequency. A spin started along $x$
precesses,

```{math}
:label: eq-te-larmor
\langle S_x\rangle=\tfrac{\hbar}{2}\cos\omega t,\quad \langle S_y\rangle=-\tfrac{\hbar}{2}\sin\omega t,\quad \langle S_z\rangle\ \text{conserved} ,
```

a classical precession of the spin vector about the field, emerging from unitary evolution (the
physics of NMR/MRI).

### Rabi oscillations

A two-level system with detuning $\delta$ driven by a coupling $\Omega$ has $H=\tfrac{\delta}{2}
\sigma_z+\tfrac{\Omega}{2}\sigma_x$. Started in the lower level, the upper-level population is

```{math}
:label: eq-rabi
\text{on resonance }(\delta=0):\ P_e(t)=\sin^2\tfrac{\Omega t}{2};\qquad \text{off resonance: }P_e^{\max}=\frac{\Omega^2}{\Omega^2+\delta^2}\ \text{at}\ \Omega_R=\sqrt{\Omega^2+\delta^2} .
```

On resonance the cycling is complete (a $\pi$-pulse, $\Omega t=\pi$, flips the state); off resonance
it is faster but shallower. Sakurai & Napolitano (§5.5) derive Rabi's formula in full; the Feynman
Lectures Vol. III reach the same physics through the ammonia maser. This is how a qubit is driven —
NMR pulses, laser excitation, single-qubit gates.

### Ehrenfest's theorem and conservation

For an observable $A$ without explicit time dependence, one line of calculus suffices: differentiate
$\langle\psi|A|\psi\rangle$ and let the Schrödinger equation supply $\partial_t|\psi\rangle$ and its
adjoint (Griffiths, Ch. 3, records the general statement),

```{math}
:label: eq-ehrenfest
\frac{d\langle A\rangle}{dt}=\frac{i}{\hbar}\langle[H,A]\rangle ,
```

so an observable that **commutes** with $H$ is **conserved** (and energy always is). The commutator
$[H,\cdot]$ plays the role of the classical Poisson bracket $\{H,\cdot\}$ of Volume II — canonical
quantization is $\{\cdot,\cdot\}\to\tfrac{1}{i\hbar}[\cdot,\cdot]$.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy.linalg import expm

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: ℏ=1. The Pauli matrices (§6.6) and spin operators S=½σ; states are unit
# complex vectors. Time evolution uses the spectral construction U=Σe^{−iE_n t}|n⟩⟨n| from
# numpy.linalg.eigh (transparent and efficient), cross-checked against scipy.linalg.expm; expectation
# values use numpy.vdot (conjugate-first, §6.1).

SIGMA_X = np.array([[0, 1], [1, 0]], dtype=complex)
SIGMA_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
SIGMA_Z = np.array([[1, 0], [0, -1]], dtype=complex)
S_X, S_Y, S_Z = SIGMA_X / 2, SIGMA_Y / 2, SIGMA_Z / 2


def evolve(H, psi0, t):
    """Evolve a state to time ``t`` under a time-independent Hamiltonian {eq}`eq-schrodinger`.

    Uses the **spectral construction** $U(t)=\\sum_n e^{-iE_nt}|n\\rangle\\langle n|$ ($\\hbar=1$):
    diagonalize $H$ with ``numpy.linalg.eigh`` to get energies $E_n$ and eigenvectors, expand
    ``psi0`` in that basis, attach the phase $e^{-iE_nt}$ to each component, and transform back. This
    is the same operator as ``scipy.linalg.expm(-1j*H*t) @ psi0`` but exposes the physics — evolution
    is a set of rotating phases — and is cheaper for many times.

    Parameters
    ----------
    H : numpy.ndarray
        A Hermitian Hamiltonian.
    psi0 : numpy.ndarray
        The initial unit state.
    t : float
        The time to evolve to.

    Returns
    -------
    numpy.ndarray
        The state $|\\psi(t)\\rangle$.
    """
    energies, basis = np.linalg.eigh(H)
    coeffs = basis.conj().T @ psi0  # components c_n = ⟨n|ψ0⟩
    return basis @ (np.exp(-1j * energies * t) * coeffs)


def expectation_value(A, psi):
    """The expectation value $\\langle A\\rangle=\\langle\\psi|A|\\psi\\rangle$ (§6.5), via ``numpy.vdot``.

    Real because $A$ is Hermitian (``numpy.vdot(psi, A @ psi).real``).
    """
    return np.vdot(psi, A @ psi).real


def expectation_trace(H, psi0, A, times):
    """The trace $\\langle A\\rangle(t)$ of an observable as the state evolves {eq}`eq-stationary`.

    Evolves ``psi0`` to each time in ``times`` with :func:`evolve` and returns the array of
    expectation values $\\langle\\psi(t)|A|\\psi(t)\\rangle$ — the laboratory's time-resolved signal.

    Parameters
    ----------
    H : numpy.ndarray
        The Hamiltonian.
    psi0 : numpy.ndarray
        The initial state.
    A : numpy.ndarray
        The observable to track.
    times : array_like
        The times at which to sample.

    Returns
    -------
    numpy.ndarray
        $\\langle A\\rangle(t)$ at each time.
    """
    return np.array([expectation_value(A, evolve(H, psi0, t)) for t in times])

## Exercise 1 — The time-evolution operator

For a given Hamiltonian $H$, construct the time-evolution operator $U(t)=e^{-iHt}$ (with
$\hbar=1$) two ways — by the matrix exponential and by the spectral construction — and verify it
solves the Schrödinger equation $i\dot U=HU$ and is unitary {eq}`eq-schrodinger`.

1. Build $U(t)$ with `scipy.linalg.expm(-1j*H*t)` and, independently, as $\sum_n
   e^{-iE_nt}|n\rangle\langle n|$ from `numpy.linalg.eigh` (energies and eigenvectors); confirm
   the two agree with `numpy.allclose`.
2. Verify $i\dot U=HU$ by approximating $\dot U$ with a centered finite difference
   $[U(t+dt)-U(t-dt)]/(2dt)$ and comparing $i\dot U$ to `H @ U`.
3. Verify $U^{\dagger}U=I$ (`U.conj().T @ U` equals `numpy.eye`).
4. Evolve a state with the `evolve` helper and confirm its norm is preserved. Frame: the dynamics
   postulate, solved.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    [schrodinger_residual, unitary_residual, norm_t - 1.0],
    [0.0, 0.0, 0.0],
    "U(t)=e^{−iHt} solves the Schrödinger equation iU̇=HU and is unitary (norm and probability conserved)",
    atol=1e-5,
)

## Exercise 2 — Stationary states

Show why energy eigenstates are special: an eigenstate of $H$ evolves by a pure phase, so every
observable's probabilities in it are frozen in time — a **stationary state** {eq}`eq-stationary`.

1. Diagonalize $H$ with `numpy.linalg.eigh` and take an eigenstate $|n\rangle$ (a column of the
   eigenvector matrix) with energy $E_n$.
2. Evolve it with the `evolve` helper to several times.
3. Confirm $|\psi(t)\rangle=e^{-iE_nt}|n\rangle$ — only a phase — by checking $|\langle n|
   \psi(t)\rangle|=1$ at every time (`numpy.abs(numpy.vdot(...))`).
4. Confirm the expectation of an arbitrary observable is time-independent in this state. Frame:
   this is why "find the energy eigenstates" is the central task — once diagonalized, evolution is
   a set of phases.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    overlaps,
    np.ones_like(overlaps),
    "an energy eigenstate is stationary — it acquires only a phase e^{−iE_nt}, so |⟨n|ψ(t)⟩|=1 and every probability is frozen",
    rtol=1e-12,
)

## Exercise 3 — Beats: the evolution of a superposition

Evolve an equal superposition of two energy eigenstates and show that an observable *not* diagonal
in the energy basis oscillates at the **Bohr frequency** $(E_2-E_1)/\hbar$ — the beating of energy
phases that is all of quantum dynamics {eq}`eq-stationary`.

1. Take a Hamiltonian with known energies (e.g. $H=\mathrm{diag}(E_1,E_2)$) and form the
   superposition $|\psi\rangle=(|1\rangle+|2\rangle)/\sqrt2$.
2. Evolve it with the `evolve` helper over a grid of times (`numpy.linspace`).
3. Compute $\langle A\rangle(t)$ with `expectation_trace` for an observable off-diagonal in the
   energy basis (e.g. $\sigma_x$).
4. Identify the oscillation frequency as $(E_2-E_1)/\hbar$ — read it off the period, or confirm
   $\langle\sigma_x\rangle(t)=\cos [(E_2-E_1)t]$. Frame: all dynamics is the interference of
   energy-eigenstate phases.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    sx_trace,
    np.cos(bohr * times_beat),
    "a superposition of two energy eigenstates beats: an off-diagonal observable oscillates at the Bohr frequency (E2−E1)/ℏ",
    rtol=1e-10,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Spin precession (Larmor)

A spin in a magnetic field along $z$, started along $x$, is evolved under the Larmor Hamiltonian
$H=-\tfrac{\omega}{2}\sigma_z$ (i.e. $-\gamma\mathbf B\cdot\mathbf S$ for $\mathbf B\parallel \hat
z$). Find $\langle S_x\rangle(t)$, $\langle S_y\rangle(t)$, $\langle S_z\rangle(t)$ and show the
spin vector **precesses** about the field at the Larmor frequency $\omega$ {eq}`eq-te-larmor`.

1. Build $H=-\tfrac{\omega}{2}\,$`SIGMA_Z`.
2. Start in $|{+}x\rangle=$`numpy.array( [1,1])/numpy.sqrt(2)`.
3. Evolve with the `evolve` helper over a grid of times and compute the three traces with
   `expectation_trace` for `S_X`, `S_Y`, `S_Z`.
4. Confirm $\langle S_x\rangle=\tfrac12\cos \omega t$, $\langle S_y\rangle=-\tfrac12\sin\omega t$,
   and $\langle S_z\rangle$ is conserved (zero) — a classical precession of the spin vector about
   the field. Frame: a classical-looking rotation emerging from unitary evolution (the physics of
   NMR).

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    [sx_prec, sy_prec],
    [0.5 * np.cos(omega * times_prec), -0.5 * np.sin(omega * times_prec)],
    "a spin precesses about the field at the Larmor frequency: ⟨Sx⟩=½cos(ωt), ⟨Sy⟩=−½sin(ωt)",
    rtol=1e-9,
)
validate.close(
    sz_prec.max() - sz_prec.min(),
    0.0,
    "the component along the field, ⟨Sz⟩, is conserved during precession (it commutes with H)",
    atol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Rabi oscillations on resonance

A two-level system driven on resonance ($\delta=0$) by a coupling $\Omega$, started in its lower
level $|0\rangle$, has $H=\tfrac{\Omega}{2}\sigma_x$. Find the probability $P_e(t)$ of finding it
in the upper level, and show it cycles completely as $\sin^2(\Omega t/2)$ {eq}`eq-rabi`.

1. Build $H=\tfrac{\Omega}{2}\,$`SIGMA_X` (resonant, $\delta=0$).
2. Start in $|0\rangle=$`numpy.array([1,0])`.
3. Evolve with the `evolve` helper and compute $P_e(t)=|\langle 1| \psi(t)\rangle|^2$ with
   `numpy.abs(numpy.vdot([0,1], psi))**2` over a grid of times.
4. Confirm $P_e(t)=\sin^2(\Omega t/2)$ — complete cycling at the Rabi frequency $\Omega$ — and
   identify the $\pi$-pulse ($\Omega t=\pi$) that fully flips the state. Frame: driving a qubit
   between its levels (a single-qubit gate).

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    P_excited,
    np.sin(Omega * times_rabi / 2) ** 2,
    "resonant Rabi oscillation: the driven two-level system cycles completely, P_excited(t)=sin²(Ωt/2)",
    rtol=1e-10,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Off-resonant (generalized) Rabi oscillations

Repeat the driven two-level problem with a **detuning** $\delta\ne0$, $H=\tfrac{\delta}
{2}\sigma_z+\tfrac{\Omega}{2}\sigma_x$. Show the oscillation becomes *faster* — at the generalized
Rabi frequency $\Omega_R=\sqrt{\Omega^2+\delta^2}$ — but *shallower*, reaching a maximum excited
population $\Omega^2/(\Omega^2+\delta^2)<1$ {eq}`eq-rabi`.

1. Build $H=\tfrac{\delta}{2}\,$`SIGMA_Z`$+\tfrac{\Omega}{2}\,$`SIGMA_X` for a few detunings
   $\delta$.
2. Evolve from $|0\rangle$ and compute $P_e(t)$ as in Exercise 5.
3. Read off the maximum of $P_e(t)$ (`numpy.max`) and confirm it equals
   $\Omega^2/(\Omega^2+\delta^2)$.
4. Confirm the oscillation frequency is $\Omega_R=\sqrt{\Omega^2+\delta^2}$ (e.g. the first return
   to $P_e=0$ at $t=2\pi/\Omega_R$). Frame: detuning reduces the transfer — resonance matters, the
   basis of spectroscopy.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    max_populations,
    predicted_max,
    "off-resonance the Rabi oscillation is faster (Ω_R=√(Ω²+δ²)) but shallower: the maximum excited population is Ω²/(Ω²+δ²) < 1",
    atol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Ehrenfest's theorem and conservation laws *(student)*

Verify Ehrenfest's theorem $d\langle A\rangle/dt=i\langle[H,A]\rangle$ ($\hbar=1$) for a chosen
observable, and use it to identify a conserved quantity — one that commutes with $H$
{eq}`eq-ehrenfest`.

1. Take the precession Hamiltonian $H=-\tfrac{\omega}{2}\sigma_z$ and the observable $A=S_x$, with
   the state evolving from $|{+}x\rangle$.
2. Compute $d\langle A\rangle/dt$ by a centered finite difference of the `expectation_trace`.
3. Compute $i\langle[H,A]\rangle$ directly (the `commutator` via `@`, then `numpy.vdot`) at the
   same times.
4. Confirm the two agree.
5. Show that $S_z$, which commutes with $H$ ($[H,S_z]=0$), is **conserved** — its expectation is
   constant. Frame: conserved means commuting with $H$; the commutator $[H,\cdot]$ is the quantum
   Poisson bracket (Volume II, canonical quantization).

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    dAdt,
    ehrenfest_rhs,
    "Ehrenfest's theorem: d⟨A⟩/dt = (i/ℏ)⟨[H,A]⟩; the rate of change of an expectation value is set by the commutator with H",
    atol=1e-4,
)
validate.check(
    sz_conserved and (sz_trace.max() - sz_trace.min()) < 1e-10,
    "an observable that commutes with H is conserved: ⟨Sz⟩ is constant under precession about z",
)

## Exercise 8 — The state in motion, and why diagonalization is everything

The Hamiltonian generates a **unitary flow** $U(t)=e^{-iHt/\hbar}$, and in its own eigenbasis that
flow is nothing but a set of rotating phases $e^{-iE_nt/\hbar}$. So the entire problem of dynamics
collapses to one task — find the energy eigenstates — after which a stationary state stands still and
a superposition beats at the Bohr frequencies. In the two-state system we watched this become a
**precessing spin** and a **driven oscillation**, the very motions behind magnetic resonance, atomic
clocks, and quantum gates. **Ehrenfest's theorem** told us which quantities survive the motion (those
commuting with $H$) and tied the commutator to the classical Poisson bracket.

**This exercise (synthesis).** No new computation: the structure is the result. "Solve the Schrödinger
equation" sounds like the hard part of quantum mechanics. It is the *easy* part — once you have the
energy eigenstates, time is just phases. The hard part, the work of the rest of the volume, is finding
them. The next notebook ([§6.8](bloch-sphere-entanglement.ipynb)) draws the geometry that makes all of this visible: the Bloch sphere,
where a qubit's state is a point, its evolution a rotation, and precession a literal spinning. Beyond
the qubit, the same exponential will evolve wave packets and orbitals — the task is always the same,
diagonalize $H$.

## Notebook summary

The dynamics postulate of [§6.5](postulates.ipynb), worked out — and the structure of all quantum motion.

- **The time-evolution operator** {eq}`eq-schrodinger`: $U(t)=e^{-iHt/\hbar}$ solves $i\hbar\dot U=HU$
  and is unitary (probability conserved); `scipy.linalg.expm` and the spectral construction $\sum_n
  e^{-iE_nt/\hbar}|n\rangle\langle n|$ agree.
- **Stationary states** {eq}`eq-stationary`: an energy eigenstate acquires only a phase, so its
  probabilities are frozen; a superposition **beats** at the Bohr frequencies $(E_n-E_m)/\hbar$. All
  dynamics is the interference of energy phases — so diagonalizing $H$ solves the motion.
- **Larmor precession** {eq}`eq-te-larmor`: $\langle S_x\rangle=\tfrac12\cos\omega t$, $\langle S_y\rangle
  =-\tfrac12\sin\omega t$, $\langle S_z\rangle$ conserved — a classical precession from unitary
  evolution (NMR).
- **Rabi oscillations** {eq}`eq-rabi`: on resonance $P_e=\sin^2(\Omega t/2)$ (complete cycling, the
  $\pi$-pulse gate); off resonance faster ($\Omega_R=\sqrt{\Omega^2+\delta^2}$) but shallower
  ($P_e^{\max}=\Omega^2/(\Omega^2+\delta^2)$).
- **Ehrenfest's theorem** {eq}`eq-ehrenfest`: $d\langle A\rangle/dt=(i/\hbar)\langle[H,A]\rangle$;
  observables commuting with $H$ are conserved, and $[H,\cdot]$ is the quantum Poisson bracket.

Solving the Schrödinger equation is the easy part — once the energy eigenstates are in hand, time is
just phases. Finding them is the rest of the volume.

## Outlook

- **The Bloch sphere ([§6.8](bloch-sphere-entanglement.ipynb))**: qubit states as points, unitary evolution as rotation, precession made
  geometric.
- **Continuous-system dynamics ([§6.10](schrodinger-on-a-computer.ipynb)–[§6.13](scattering-tunneling.ipynb))**: evolving wave packets, the time-dependent Schrödinger
  equation by split-step and Crank–Nicolson.
- **Driven transitions and Fermi's golden rule ([§6.24](time-dependent-perturbation.ipynb))**: Rabi physics in the many-level setting,
  time-dependent perturbation theory.
- **The classical limit**: the commutator as the Poisson bracket (Volume II), Ehrenfest $\to$ Newton —
  canonical quantization $\{\cdot,\cdot\}\to\tfrac{1}{i\hbar}[\cdot,\cdot]$.
- **Cross-reference** [§6.2](operators-spectral-theorem.ipynb) (the exponential map), [§6.3](dirac-notation-spectral-decomposition.ipynb) (functions of operators), [§6.5](postulates.ipynb) (the dynamics
  postulate), [§6.6](pauli-uncertainty.ipynb) (commutators, conservation), Volume II (Poisson brackets), and forward to [§6.8](bloch-sphere-entanglement.ipynb),
  [§6.10](schrodinger-on-a-computer.ipynb)–[§6.13](scattering-tunneling.ipynb), [§6.24](time-dependent-perturbation.ipynb).

In [ ]:
from ecp.style import footer

footer()